# Chương 4.4 — Sensitivity Analysis của Cụm D: TrustRank Power Iteration

4 tham số: Alpha (damping), Epsilon (ngưỡng hội tụ), MaxIter, SeedThreshold.

Output từ `sensitivity bench --cluster=D`. Vì power iteration là thuật toán lặp **xác định** theo (graph, Alpha), Sobol bị thay bằng **convergence study** (số vòng lặp tới hội tụ theo Alpha).

> **Phát hiện chính:**
> - **Alpha** chi phối mạnh nhất cả độ lớn điểm lẫn thời gian hội tụ; số vòng lặp bùng nổ khi α→1 (α=0.99 không hội tụ trong 100 vòng).
> - **SeedThreshold** có hiệu ứng *ngưỡng* (cliff): dưới ~68 tập seed nở ra làm đảo thứ hạng (ρ≈0.76); từ 68 trở lên ổn định (ρ=1.0). Default 80 nằm an toàn trong vùng ổn định.
> - **MaxIter** ≥ ~19 và **Epsilon** nhỏ → đã hội tụ hoàn toàn, không ảnh hưởng thứ hạng. Default (MaxIter=30, Eps=1e-4) nằm trong vùng hội tụ.

In [ ]:
import sys
sys.path.insert(0, '..')
from lib import setup_thesis_style, save_figure, load_oat, load_tornado

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

setup_thesis_style()

SNAP_ID = "snap_20260603_014800"
OUTPUT_DIR = f"../output/{SNAP_ID}"
CLUSTER = "D"

In [ ]:
conv = pd.read_csv(f"{OUTPUT_DIR}/cluster_D/convergence.csv")
MAXITER = 100  # cap used by the convergence run
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(conv["alpha"], conv["iterations"], marker="o", color="#1f77b4")
ax.axhline(MAXITER, ls="--", color="#d62728", lw=0.8, label=f"MaxIter cap = {MAXITER}")
ax.set_xlabel("Alpha (damping factor)")
ax.set_ylabel("Số vòng lặp tới hội tụ")
ax.set_title("Fig 4.4.1 — Power iteration: số vòng lặp hội tụ theo Alpha")
ax.legend(loc="upper left")
ax.grid(True, alpha=0.3)
fig.tight_layout()
save_figure(fig, "4_4_1_alpha_convergence")
plt.show()

In [ ]:
oat = load_oat(OUTPUT_DIR, CLUSTER)
params = oat["param"].unique()
fig, axes = plt.subplots(nrows=2, ncols=2, figsize=(10, 7), sharey=False)
axes = axes.flatten()
for ax, p in zip(axes, params):
    sub = oat[oat["param"] == p]
    ax.plot(sub["value"], sub["spearman"], marker="o", color="#1f77b4", label="Spearman ρ")
    ax2 = ax.twinx()
    ax2.plot(sub["value"], sub["mean"], marker="s", color="#ed7d31", label="mean trust")
    ax.axhline(0.95, ls="--", color="grey", lw=0.5)
    ax.set_title(p)
    ax.set_xlabel("value")
    ax.set_ylabel("Spearman ρ", color="#1f77b4")
    ax2.set_ylabel("mean trust", color="#ed7d31")
    ax.set_ylim(0, 1.05)
fig.suptitle("Fig 4.4.2 — OAT rank stability + mean trust (Cluster D)")
fig.tight_layout()
save_figure(fig, "4_4_2_oat")
plt.show()

In [ ]:
tor = load_tornado(OUTPUT_DIR, CLUSTER)
tor["total"] = tor["delta_low"] + tor["delta_high"]
tor = tor.sort_values("total")
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.barh(tor["param"], -tor["delta_low"], color="#5b9bd5", label="Δ low (−)")
ax.barh(tor["param"], tor["delta_high"], color="#ed7d31", label="Δ high (+)")
ax.axvline(0, color="black", lw=0.5)
ax.set_title("Fig 4.4.3 — Tornado: |Δ mean trust| theo tham số (Cluster D)")
ax.set_xlabel("Δ điểm trust trung bình")
ax.legend(loc="lower right")
fig.tight_layout()
save_figure(fig, "4_4_3_tornado")
plt.show()

In [ ]:
grid = pd.read_csv(f"{OUTPUT_DIR}/cluster_D/grid.csv")
param_cols = [c for c in grid.columns if c not in {"combo_id", "spearman", "kendall", "mean", "std"}]
print("Top-3 grid parameters:", param_cols)
if "Alpha" in param_cols and "SeedThreshold" in param_cols:
    third = [c for c in param_cols if c not in ("Alpha", "SeedThreshold")][0]
    median3 = grid[third].median()
    sub = grid[np.isclose(grid[third], median3, rtol=0.01)]
    pivot = sub.pivot_table(index="Alpha", columns="SeedThreshold", values="spearman")
    fig, ax = plt.subplots(figsize=(7, 5))
    sns.heatmap(pivot, cmap="viridis", ax=ax)
    ax.set_title(f"Fig 4.4.4 — Spearman ρ over (Alpha, SeedThreshold) at {third}={median3:g}")
    fig.tight_layout()
    save_figure(fig, "4_4_4_alpha_seed_heatmap")
    plt.show()
else:
    print(f"Alpha or SeedThreshold not in top-3 ({param_cols}); skipping heatmap")